# ALE 3D CNN NeuroVLM Experiment

Runs the DiFuMo-compatible and atlas-free ALE 3D CNN experiments with the reusable `experiments/train_ale_cnn.py` entrypoint.

In [ ]:
import os, platform, subprocess, sys
print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)
print('Platform:', platform.platform())

In [ ]:
# Dependency install. PyG is not required for the dense ALE CNN path.
!pip -q install nilearn nibabel huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn

In [ ]:
# Repo clone/pull for Colab. Skip this cell when running inside an existing checkout.
REPO_URL = 'https://github.com/ryan-hammonds/neurovlm.git'
REPO_DIR = '/content/neurovlm'
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
os.chdir(REPO_DIR)
!pip -q install -e '.[viz,notebook,metrics]'
print(os.getcwd())

In [ ]:
# Choose experiment mode.
MODE = 'atlas_free'          # or 'difumo_compatible'
MODEL = 'ale_3dcnn'          # or 'ale_flat_mlp'
EPOCHS = 20
BATCH_SIZE = 16
RUN_DIR = f'runs/{MODEL}_{MODE}_colab'
CACHE_FILE = f'data/ale_caches/{MODE}_4mm_fwhm9_crop_float16.pt'
print(MODE, MODEL, RUN_DIR)

In [ ]:
# Build/load cache and show basic data shapes before training.
import sys, numpy as np, torch
sys.path.insert(0, 'src')
from neurovlm.gnn.ale_dataset import ALEPreprocessConfig, build_or_load_ale_cache, ALEVolumeDataset
cfg = ALEPreprocessConfig(mode=MODE, kernel_fwhm_mm=9.0, resolution_mm=4.0, crop_to_brain=True)
payload = build_or_load_ale_cache(CACHE_FILE, cfg)
ds = ALEVolumeDataset.from_cache(payload)
print('cache volumes:', tuple(payload['volumes'].shape), payload['volumes'].dtype)
print('aligned dataset:', len(ds), 'input shape:', ds.input_shape)
print('metadata:', payload['metadata'])

In [ ]:
# Visual sanity check: middle slices from a few ALE maps.
import matplotlib.pyplot as plt
n = min(4, len(ds))
fig, axes = plt.subplots(n, 3, figsize=(8, 2.5*n))
for row in range(n):
    vol = ds[row]['volume'][0].numpy()
    slices = [vol[vol.shape[0]//2], vol[:, vol.shape[1]//2, :], vol[:, :, vol.shape[2]//2]]
    for col, sl in enumerate(slices):
        ax = axes[row, col] if n > 1 else axes[col]
        ax.imshow(np.rot90(sl), cmap='magma')
        ax.axis('off')
plt.tight_layout()

In [ ]:
# Train and evaluate. Outputs include checkpoints, history, metrics, recall curve, diagnostics, and comparison CSV/JSONL.
!python experiments/train_ale_cnn.py \
  --mode $MODE \
  --model $MODEL \
  --epochs $EPOCHS \
  --batch-size $BATCH_SIZE \
  --cache-file $CACHE_FILE \
  --run-dir $RUN_DIR \
  --checkpoint-dir $RUN_DIR/checkpoints \
  --resolution-mm 4 \
  --kernel-fwhm-mm 9 \
  --base-channels 16 \
  --num-blocks 3 \
  --out-dim 384 \
  --amp \
  --umap

In [ ]:
# Load final results and plots.
import json, pandas as pd
from IPython.display import Image, display
with open(f'{RUN_DIR}/eval_results.json') as f:
    results = json.load(f)
print(pd.DataFrame(results).T[['mean_auc','mean_recall@1','mean_recall@5','mean_recall@10','mean_recall@50','mean_mrr','mean_median_rank','mean_random_recall@10']])
for png in ['training_curves.png', 'recall_curve.png', 'umap_diagnostics.png']:
    path = f'{RUN_DIR}/{png}'
    if os.path.exists(path):
        display(Image(path))